# Task 4 — Benchmark: EKF · Robust EKF · KalmanNet · existing/proposed Robust KalmanNet

Reproducible comparison of state estimators on the **Synthetic Non-Linear model**
(KalmanNet paper, Sec IV-C), under **full** and **partial** (model-mismatch) information.
Metric: per-sequence posterior state MSE (dB) on a shared test set.

## How to use
Edit **only the Configuration cell** below, then run the notebook top-to-bottom
(*Kernel → Restart & Run All*). No other cell needs manual editing.

## Configuration parameters
| option | meaning |
|---|---|
| `RUN_MODE` | `"train"` = fit the trainable models, save checkpoints, then evaluate. `"eval"` = load existing checkpoints and evaluate only, never retrain. |
| `CONFIG` | `"FAST"` = small/quick (development smoke). `"FULL"` = larger/converged (numbers for the report). |
| `SEED` | global random seed (reproducibility). |
| `CKPT_DIR` | folder for checkpoints + generated data (kept separate from the project's own files). |
| `TRAIN_KALMANNET` | include KalmanNet in the benchmark: in `train` mode it is (re)trained and its checkpoint saved; in `eval` mode its checkpoint is loaded. `False` skips KalmanNet entirely (both modes). |
| `TRAIN_PROPOSED` | same, for the proposed Robust KalmanNet. |
| `INCLUDE_TEACHER` | include the teacher (MLP original) network: run **its own training recipe** and evaluate it. Its net is *not* gradient-trainable, so this demonstrates a no-op training. `False` = skip. |

## Models
- **EKF**, **Robust EKF** — analytic filters, **evaluation only** (no training). Robust EKF is reported at its best tolerance `c` from a sweep.
- **KalmanNet**, **proposed Robust KalmanNet** — neural, **trainable** (train mode fits them and saves a checkpoint; eval mode loads their checkpoints).
- **existing Robust KalmanNet** — professor's MLP original network, evaluated **untrained** (as delivered).
- **teacher Robust KalmanNet** — the same MLP original run through **its own training recipe** (`main_Robust_KNet_original`). It is **not gradient-trainable** (its `c` reaches the loss only via a non-differentiable theta bisection, and its feedback is detached), so the notebook prints `Δw ≈ 0`, plots its flat curve, and evaluates it at both zero-init and aligned init.

## Train vs. eval mode
- **train**: the enabled trainable models are fit on the true-model data, their checkpoints are written to `CKPT_DIR` (`kalmannet_<scen>.pt`, `proposed_<scen>.pt`), and every enabled model is evaluated afterward.
- **eval**: no training occurs; the enabled trainable models are loaded from `CKPT_DIR`. If a required checkpoint is missing, the notebook stops with a clear error (naming the missing file and how to generate it) instead of silently retraining.

See `docs/TASK4_DESIGN.md` for the design rationale (aligned init, the `f`-only mismatch, the expected qualitative orderings).

In [1]:
import os, math, time, random, shutil
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt

import Simulations.config as config
from Simulations.Synthetic_NL_model.parameters import Q_structure, R_structure, m, n, m1x_0, m2x_0, f, h
from Simulations.Extended_sysmdl import SystemModel
from Simulations.utils import DataGen
from Filters.EKF_test import EKFTest
from RobustKalmanPY.robust_kalman import RobustKalman                              # REKF + proposed (use_nn)
from RobustKalmanPY.robust_kalman_original import RobustKalman as RobustKalmanOrig # existing Robust KalmanNet
from Pipelines.Pipeline_EKF import Pipeline_EKF
from KNet.KalmanNet_nn import KalmanNetNN

crit = nn.MSELoss()
def to_dB(x): return 10.0 * math.log10(x)   # seed is set in the Configuration cell below

## Configuration

Edit **only** the `USER CONFIGURATION` block in the cell below; everything under
`derived settings` is computed automatically from it.

In [ ]:
# ===================== USER CONFIGURATION (edit only this block) =====================
RUN_MODE        = "train"      # "train": fit trainable models, save checkpoints, then evaluate
                               # "eval" : load checkpoints and evaluate only (never retrain)
CONFIG          = "FAST"       # "FAST": quick smoke   |   "FULL": converged, for reported numbers
SEED            = 0            # global random seed
CKPT_DIR        = "Results_task4/"   # checkpoints + generated data (kept out of the project files)
TRAIN_KALMANNET = True         # include KalmanNet: train+save in train mode, load in eval mode (False = skip)
TRAIN_PROPOSED  = True         # include proposed Robust KalmanNet: train+save / load (False = skip)
INCLUDE_TEACHER = True         # include the teacher (MLP original): run its training loop + eval (False = skip)
# ====================================================================================

# ------------------------------ derived settings (do not edit) ----------------------
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
os.makedirs(CKPT_DIR, exist_ok=True)
path_results = CKPT_DIR
DataFile     = os.path.join(CKPT_DIR, "data_task4.pt")
device       = torch.device("cpu")

args = config.general_settings(); args.use_cuda = False; args.randomLength = False
args.T, args.T_test = 100, 100
args.n_batch = 20; args.lr, args.wd = 1e-3, 1e-4
args.in_mult_KNet, args.out_mult_KNet = 5, 40

gru_hidden_size = 16 # diminuendo da 64 a 16 il training/validation ha un andamento che converge più lentamente 
tbptt_size = 50
lr_rtk = 3e-3

if CONFIG == "FULL":
    args.N_E, args.N_CV, args.N_T = 100, 20, 40
    args.n_steps = 200                                          # KalmanNet training steps
    PROP = dict(epochs=200, n_batch=10, tbptt=tbptt_size, lr=lr_rtk, wd=1e-4, gru_hidden=gru_hidden_size)
elif CONFIG == "FAST":
    args.N_E, args.N_CV, args.N_T = 40, 10, 20
    args.n_steps = 30
    PROP = dict(epochs=20, n_batch=8, tbptt=tbptt_size, lr=lr_rtk, wd=1e-4, gru_hidden=gru_hidden_size)
else:
    raise ValueError(f"CONFIG must be 'FAST' or 'FULL', got {CONFIG!r}")

# teacher (MLP original) recipe: mirrors main_Robust_KNet_original (Adam on ALL params, wd=1e-3)
TEACHER_HIDDEN = [20, 20, 20, 20, 20]
TEACHER = dict(epochs=args.n_steps, lr=1e-3, wd=1e-3)

# noise (Sec IV-C): nu = q2/r2 = -20 dB, R = 0.1 I
r2 = torch.tensor([0.1]); vv = 10 ** (-20 / 10); q2 = torch.mul(vv, r2)
Q = q2[0] * Q_structure; R = r2[0] * R_structure

# aligned-init policy: filter x0 = generative init (off the x=0 trap); common initial covariance
X0 = torch.ones(m, 1)                          # aligned filter init
M2 = 1e-3 * torch.eye(m)                       # common initial covariance (matches REKF's V_prev)
CS = [1e-6, 1e-2, 0.05, 0.1, 0.2, 0.5, 1.0]    # Robust EKF tolerance grid (1e-6 ~ EKF; c>=2 diverges)

print(f"RUN_MODE={RUN_MODE}  CONFIG={CONFIG}  seed={SEED}  ckpt={CKPT_DIR}")
print(f"  N_E={args.N_E} N_T={args.N_T}  KNet steps={args.n_steps}  proposed epochs={PROP['epochs']}"
      f"  | aligned x0=ones, m2x0=1e-3 I")

RUN_MODE=train  CONFIG=FULL  seed=0  ckpt=Results_task4/
  N_E=100 N_T=40  KNet steps=200  proposed epochs=200  | aligned x0=ones, m2x0=1e-3 I


## System model + shared data

The TRUE model generates the data. The PARTIAL model mismatches `f` only
(`alpha=beta=1, phi=delta=0` -> `sin(x)`); `h` is unchanged; the data is always from
the TRUE model. All filters use the aligned init.

In [3]:
# TRUE (full-information) model
sys_full = SystemModel(f, Q, h, R, args.T, args.T_test, m, n)
sys_full.InitSequence(m1x_0, m2x_0)                 # generative init for DATA generation
sys_full.alpha, sys_full.beta, sys_full.phi, sys_full.delta = 0.9, 1.1, 0.1*math.pi, 0.01
sys_full.a, sys_full.b, sys_full.c = 1, 1, 0

DataGen(args, sys_full, DataFile)
train_input, train_target, cv_input, cv_target, test_input, test_target = torch.load(DataFile, map_location=device)[:6]

# PARTIAL model
def f_partial(x): return torch.sin(x)
sys_part = SystemModel(f_partial, Q, h, R, args.T, args.T_test, m, n)
sys_part.InitSequence(m1x_0, m2x_0)
sys_part.alpha, sys_part.beta, sys_part.phi, sys_part.delta = 1.0, 1.0, 0.0, 0.0
sys_part.a, sys_part.b, sys_part.c = 1, 1, 0

# apply aligned FILTER init + common covariance to both models
for s in (sys_full, sys_part):
    s.m1x_0 = X0.clone(); s.m2x_0 = M2.clone()
print('train', tuple(train_input.shape), ' test', tuple(test_input.shape),
      f" | true f(0)={0.9*math.sin(0.1*math.pi)+0.01:.4f}  partial f(0)=0  h'(0)=0")

train (100, 2, 100)  test (40, 2, 100)  | true f(0)=0.2881  partial f(0)=0  h'(0)=0


/var/folders/gm/b8zghr4d7q98yylv3k_5bm5w0000gn/T/ipykernel_57365/1372887799.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train_input, train_target, cv_input, cv_targe

## Metric & per-method helpers (posterior MSE -> dB, shared test set)

In [4]:
results  = {}
history  = {}     # (model, scenario) -> {'train_dB': [...per epoch...], 'val_dB': [...]}  (train mode only)
def rec(method, scenario, mdB, extra=''):
    results.setdefault(method, {})[scenario] = mdB
    print(f'  {method:22s} [{scenario:7s}]  MSE = {mdB:+8.3f} dB   {extra}')

# ---- checkpoint helpers (train mode writes them, eval mode requires them) ----
def ckpt_path(model, scen):
    return os.path.join(CKPT_DIR, f'{model}_{scen}.pt')

def require_ckpt(path, model, scen):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"[RUN_MODE='eval'] checkpoint for {model} ({scen}) not found:\n    {path}\n"
            f"Generate it by running the notebook once with RUN_MODE='train' (and the matching "
            f"training flag enabled), then switch RUN_MODE back to 'eval'.")

# ---- analytic filters (evaluation only, no checkpoints) ----
def ekf_dB(sysmdl):
    out = EKFTest(args, sysmdl, test_input, test_target)     # out[0] = per-seq posterior MSE
    return to_dB(float(out[0].mean()))

def rekf_sweep(sysmdl):
    row = {}
    for c in CS:
        rk = RobustKalman(sysmdl, test_input[0], float(c), False, False, sl_model=0)  # use_nn=False
        mses = []
        for k in range(test_input.shape[0]):
            rk.c = torch.tensor(float(c)); rk.y = test_input[k]; rk.fnREKF()
            mses.append(crit(rk.Xn_out, test_target[k]).item())
        row[c] = to_dB(float(np.mean(mses)))
    return row, min(row, key=row.get)

def existing_rknet_dB(sysmdl):
    net = RobustKalmanOrig(sysmdl, test_input[0], 1e-3, False, True,
                           input_feat_mode=3, hidden_layers=[20,20,20,20,20], sl_model=0).nn
    mses = []
    for k in range(test_input.shape[0]):
        rk = RobustKalmanOrig(sysmdl, test_input[k], 1e-3, False, True,
                              input_feat_mode=3, hidden_layers=[20,20,20,20,20], sl_model=0)
        rk.nn = net; rk.nn.previous_output = torch.zeros(1, 1)
        with torch.no_grad(): rk.fnREKF(train=False)
        mses.append(crit(rk.Xn, test_target[k]).item())
    return to_dB(float(np.mean(mses)))

# ---- KalmanNet: train+save (train mode) or load (eval mode), then evaluate ----
def kalmannet_run(sysmdl, scen):
    path = ckpt_path('kalmannet', scen)
    pipe = Pipeline_EKF(f'task4-knet-{scen}', 'KNet', 'KNet'); pipe.setssModel(sysmdl)
    if RUN_MODE == 'train':
        torch.manual_seed(SEED)
        KNet = KalmanNetNN(); KNet.NNBuild(sysmdl, args)
        pipe.setModel(KNet); pipe.setTrainingParams(args)
        # NNTrain runs the epoch loop / optimizer / backward (Pipelines/Pipeline_EKF.py) and
        # returns the per-epoch loss curves: [cv_linear, cv_dB, train_linear, train_dB].
        cv_lin, cv_dB, tr_lin, tr_dB = pipe.NNTrain(sysmdl, cv_input, cv_target,
                                                    train_input, train_target, path_results)  # on TRUE data
        history[('KalmanNet', scen)] = {'train_dB': tr_dB.tolist(), 'val_dB': cv_dB.tolist()}  # keep the curve
        shutil.copyfile(os.path.join(path_results, 'best-model.pt'), path)                      # persist checkpoint
    else:
        require_ckpt(path, 'KalmanNet', scen)
        pipe.setModel(torch.load(path, map_location=device)); pipe.setTrainingParams(args)
    out = pipe.NNTest(sysmdl, test_input, test_target, path_results,
                      load_model=True, load_model_path=path)
    return to_dB(float(out[1]))

# ---- proposed Robust KalmanNet: train+save (train mode) or load (eval mode), then evaluate ----
def proposed_run(sysmdl, scen):
    path = ckpt_path('proposed', scen)
    prop = RobustKalman(sysmdl, train_input[0], 1e-3, False, True,
                        input_feat_mode=3, gru_hidden_size=PROP['gru_hidden'], sl_model=0)  # use_nn=True (GRU)
    if RUN_MODE == 'train':
        torch.manual_seed(SEED)
        # exclude the output layer from weight decay so it is not pulled to the sigmoid default
        decay   = [p for nm, p in prop.nn.named_parameters() if 'output_layer' not in nm]
        nodecay = [p for nm, p in prop.nn.named_parameters() if 'output_layer' in nm]
        opt = optim.Adam([{'params': decay, 'weight_decay': PROP['wd']},
                          {'params': nodecay, 'weight_decay': 0.0}], lr=PROP['lr'])
        tr_hist, val_hist = [], []
        for ep in range(PROP['epochs']):                          # === optimization loop (epochs) ===
            idx = random.sample(range(train_input.shape[0]), min(PROP['n_batch'], train_input.shape[0]))
            opt.zero_grad(); losses = []
            for j in idx:                                          # mini-batch BPTT on posterior state-MSE
                prop.y = train_input[j]
                prop.fnREKF(train=True, reset=True, bptt_truncation=PROP['tbptt'])
                losses.append(crit(prop.Xn_out, train_target[j]))
            loss = torch.stack(losses).mean()                     # training loss (this epoch)
            loss.backward()                                       # backward pass
            torch.nn.utils.clip_grad_norm_(prop.nn.parameters(), 1.0); opt.step()  # optimizer step
            # ---- record + show the loss (validation pass is no_grad -> does not affect the update) ----
            with torch.no_grad():
                vmse = []
                for j in range(cv_input.shape[0]):
                    prop.y = cv_input[j]; prop.fnREKF(train=False, reset=True)
                    vmse.append(crit(prop.Xn_out, cv_target[j]).item())
            tr_hist.append(to_dB(loss.item())); val_hist.append(to_dB(float(np.mean(vmse))))
            print(f"  proposed[{scen:7s}] epoch {ep+1:3d}/{PROP['epochs']}   "
                  f"train {tr_hist[-1]:+7.3f} dB   val {val_hist[-1]:+7.3f} dB")
        history[('proposed Robust KNet', scen)] = {'train_dB': tr_hist, 'val_dB': val_hist}   # keep the curve
        torch.save(prop.nn.state_dict(), path)                    # persist checkpoint (state_dict only)
    else:
        require_ckpt(path, 'proposed Robust KalmanNet', scen)
        prop.nn.load_state_dict(torch.load(path, map_location=device))
    mses, cbar = [], []
    for k in range(test_input.shape[0]):
        prop.y = test_input[k]
        with torch.no_grad(): prop.fnREKF(train=False, reset=True)
        mses.append(crit(prop.Xn_out, test_target[k]).item())
        cbar.append(float(torch.stack([c.reshape(()) for c in prop.c_array]).mean()))
    return to_dB(float(np.mean(mses))), float(np.mean(cbar))

## 1. EKF + Robust EKF (`c`-sweep) - full + partial

In [5]:
cstar = {}
for scen, sysmdl in [('full', sys_full), ('partial', sys_part)]:
    rec('EKF', scen, ekf_dB(sysmdl))
    row, bc = rekf_sweep(sysmdl); cstar[scen] = bc
    print('      REKF sweep [%s]: ' % scen + '  '.join(f'c{c}:{row[c]:+.2f}' for c in CS)
          + f'   -> c*={bc}')
    rec('Robust EKF', scen, row[bc], f'(c*={bc})')

Extended Kalman Filter - MSE LOSS: tensor(-29.7125) [dB]
Extended Kalman Filter - STD: tensor(0.4557) [dB]
Inference Time: 0.5324819087982178
  EKF                    [full   ]  MSE =  -29.712 dB   
      REKF sweep [full]: c1e-06:-29.29  c0.01:-29.29  c0.05:-29.29  c0.1:-29.28  c0.2:-29.26  c0.5:-29.18  c1.0:-29.01   -> c*=0.01
  Robust EKF             [full   ]  MSE =  -29.295 dB   (c*=0.01)
Extended Kalman Filter - MSE LOSS: tensor(-8.6797) [dB]
Extended Kalman Filter - STD: tensor(0.1083) [dB]
Inference Time: 0.40360426902770996
  EKF                    [partial]  MSE =   -8.680 dB   
      REKF sweep [partial]: c1e-06:-8.74  c0.01:-9.57  c0.05:-10.66  c0.1:-11.46  c0.2:-12.48  c0.5:-13.88  c1.0:-14.62   -> c*=1.0
  Robust EKF             [partial]  MSE =  -14.622 dB   (c*=1.0)


## 2. KalmanNet — aligned init, full + partial

In `train` mode it is fit on the TRUE data and its checkpoint saved; in `eval` mode the
checkpoint is loaded. Disabled (`TRAIN_KALMANNET=False`) → skipped.

In [6]:
if TRAIN_KALMANNET:
    for scen, sysmdl in [('full', sys_full), ('partial', sys_part)]:
        rec('KalmanNet', scen, kalmannet_run(sysmdl, scen))
else:
    print('  KalmanNet             [skipped]  (TRAIN_KALMANNET=False)')

Epoch 1/200, MSE Training: 0.0048, MSE Validation: 0.0028
Epoch 2/200, MSE Training: 0.0028, MSE Validation: 0.0019
Epoch 3/200, MSE Training: 0.0020, MSE Validation: 0.0015
Epoch 4/200, MSE Training: 0.0015, MSE Validation: 0.0014
Epoch 5/200, MSE Training: 0.0014, MSE Validation: 0.0015
Epoch 6/200, MSE Training: 0.0015, MSE Validation: 0.0016
Epoch 7/200, MSE Training: 0.0017, MSE Validation: 0.0017
Epoch 8/200, MSE Training: 0.0018, MSE Validation: 0.0017
Epoch 9/200, MSE Training: 0.0018, MSE Validation: 0.0016
Epoch 10/200, MSE Training: 0.0017, MSE Validation: 0.0015
Epoch 11/200, MSE Training: 0.0016, MSE Validation: 0.0014
Epoch 12/200, MSE Training: 0.0015, MSE Validation: 0.0013
Epoch 13/200, MSE Training: 0.0014, MSE Validation: 0.0012
Epoch 14/200, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 15/200, MSE Training: 0.0012, MSE Validation: 0.0012
Epoch 16/200, MSE Training: 0.0012, MSE Validation: 0.0011
Epoch 17/200, MSE Training: 0.0012, MSE Validation: 0.0011
Epoch 

/Users/llp/Desktop/RT-KalmanNet/CODE/RT_KFNET/Pipelines/Pipeline_EKF.py:275: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model = torch.load(load_model_path, map_locat

  KalmanNet              [full   ]  MSE =  -29.698 dB   
Epoch 1/200, MSE Training: 1.2709, MSE Validation: 0.4933
Epoch 2/200, MSE Training: 0.5018, MSE Validation: 0.0753
Epoch 3/200, MSE Training: 0.0734, MSE Validation: 0.0397
Epoch 4/200, MSE Training: 0.0395, MSE Validation: 0.0271
Epoch 5/200, MSE Training: 0.0283, MSE Validation: 0.0213
Epoch 6/200, MSE Training: 0.0224, MSE Validation: 0.0184
Epoch 7/200, MSE Training: 0.0185, MSE Validation: 0.0168
Epoch 8/200, MSE Training: 0.0173, MSE Validation: 0.0160
Epoch 9/200, MSE Training: 0.0164, MSE Validation: 0.0157
Epoch 10/200, MSE Training: 0.0153, MSE Validation: 0.0156
Epoch 11/200, MSE Training: 0.0159, MSE Validation: 0.0157
Epoch 12/200, MSE Training: 0.0159, MSE Validation: 0.0159
Epoch 13/200, MSE Training: 0.0167, MSE Validation: 0.0162
Epoch 14/200, MSE Training: 0.0166, MSE Validation: 0.0165
Epoch 15/200, MSE Training: 0.0175, MSE Validation: 0.0168
Epoch 16/200, MSE Training: 0.0169, MSE Validation: 0.0171
Epoch 17

## 3. Existing Robust KalmanNet (professor's original; untrained by construction)

In [7]:
rec('existing Robust KNet', 'full',    existing_rknet_dB(sys_full))
rec('existing Robust KNet', 'partial', existing_rknet_dB(sys_part))

  existing Robust KNet   [full   ]  MSE =  -29.168 dB   
  existing Robust KNet   [partial]  MSE =  -13.908 dB   


## 4. Proposed Robust KalmanNet (GRU-learned `c_t`, mini-batch BPTT)

In `train` mode it is fit with the posterior state-MSE loss on the TRUE data and its
checkpoint saved (state_dict); in `eval` mode the checkpoint is loaded. Disabled
(`TRAIN_PROPOSED=False`) → skipped. The v3 head is `sigmoid -> c in (0,1)`: if the
beneficial tolerance is `>= 1` (the sweep above found `c*` near 1), `c_t` will saturate
near the ceiling — see `docs/TASK4_DESIGN.md` for the `c_range`/`leo` alternative if you
want to lift that cap.

In [ ]:
if TRAIN_PROPOSED:
    for scen, sysmdl in [('full', sys_full), ('partial', sys_part)]:
        mdB, cavg = proposed_run(sysmdl, scen)
        rec('proposed Robust KNet', scen, mdB, f'(mean learned c = {cavg:.3f})')
else:
    print('  proposed Robust KNet  [skipped]  (TRAIN_PROPOSED=False)')

  proposed[full   ] epoch   1/200   train -29.382 dB   val -29.304 dB
  proposed[full   ] epoch   2/200   train -29.337 dB   val -29.305 dB
  proposed[full   ] epoch   3/200   train -29.182 dB   val -29.307 dB
  proposed[full   ] epoch   4/200   train -29.280 dB   val -29.308 dB
  proposed[full   ] epoch   5/200   train -29.342 dB   val -29.309 dB
  proposed[full   ] epoch   6/200   train -29.098 dB   val -29.310 dB
  proposed[full   ] epoch   7/200   train -29.098 dB   val -29.312 dB
  proposed[full   ] epoch   8/200   train -29.259 dB   val -29.312 dB
  proposed[full   ] epoch   9/200   train -29.702 dB   val -29.313 dB
  proposed[full   ] epoch  10/200   train -29.041 dB   val -29.314 dB
  proposed[full   ] epoch  11/200   train -29.134 dB   val -29.315 dB
  proposed[full   ] epoch  12/200   train -29.253 dB   val -29.315 dB
  proposed[full   ] epoch  13/200   train -29.407 dB   val -29.316 dB
  proposed[full   ] epoch  14/200   train -29.205 dB   val -29.317 dB
  proposed[full   ] 

## 4b. Teacher Robust KalmanNet (MLP original) — training attempt + evaluation

The professor's original network (`robust_kalman_original.py`, the MLP with
`hidden_layers=[20,20,20,20,20]`) run through **its own training recipe**
(`main_Robust_KNet_original`: Adam on all params, `wd=1e-3`, one full sequence/epoch,
zero-init).

**This net is not gradient-trainable.** Its tolerance `c` reaches the loss only through
the non-differentiable theta **bisection**, and its recurrent feedback is detached, so
backprop cannot update the weights: on the **full** model every parameter comes back with
`grad=None`; on the **partial** model the `train=True` forward isn't even autograd-safe and
`backward` raises (caught below). Either way the Adam step is a no-op. The cell **proves it**:
it prints the weight change `dw = 0` and its loss curve does not descend. (What looked like
"training" in the reference was best-CV *selection* over fresh random data, not learning.)
The net is then evaluated on the shared test set at the teacher's `[0init]` (its original
regime; note the partial + zero-init trap `f(0)=0, h'(0)=0` freezes the estimate) and at our
`[aligned]` init — where it sits right next to the untrained *existing Robust KNet*.

In [ ]:
# ---- Teacher (MLP original) training ATTEMPT + evaluation ---------------------------
# The MLP net's tolerance c enters the filter only through the non-differentiable theta
# bisection, and its recurrent feedback is detached -> backprop cannot reach the weights
# (verified: on the full model every param gets grad=None; on the partial model the
# train=True forward is not even autograd-safe and backward raises). We still run the
# teacher's own recipe (Adam on ALL params, wd=1e-3, one full sequence/epoch, zero-init)
# to DEMONSTRATE the no-op: the printed weight change dw is 0 either way, and the loss
# curve does not descend. We then evaluate the (unchanged) net on the shared test set at
# both the teacher's zero-init and our aligned init.
def teacher_run(sysmdl, scen):
    torch.manual_seed(SEED)
    net = RobustKalmanOrig(sysmdl, train_input[0], 1e-3, False, True,
                           input_feat_mode=3, hidden_layers=TEACHER_HIDDEN, sl_model=0).nn
    opt = optim.Adam(net.parameters(), lr=TEACHER['lr'], weight_decay=TEACHER['wd'])   # all params
    w0 = torch.cat([p.detach().flatten() for p in net.parameters()]).clone()

    saved_init = sysmdl.m1x_0
    sysmdl.m1x_0 = torch.zeros(m, 1)                    # teacher trains at zero-init
    tr_hist, val_hist = [], []
    note = 'grad~0 (no update)'
    for ep in range(TEACHER['epochs']):                # === teacher optimization loop (epochs) ===
        j = ep % train_input.shape[0]                  # one full sequence per epoch (no TBPTT)
        rk = RobustKalmanOrig(sysmdl, train_input[j], 1e-3, False, True,
                              input_feat_mode=3, hidden_layers=TEACHER_HIDDEN, sl_model=0)
        rk.nn = net; net.previous_output = torch.zeros(1, 1)
        opt.zero_grad()
        out = rk.fnREKF(train=True)                    # [Xrekf, c_array, V, t]
        loss = crit(out[0][:, :-1], train_target[j])   # teacher's own loss (on the propagated state)
        try:
            loss.backward(); opt.step()                # weights do not move: grads ~0 (full) ...
        except RuntimeError:
            note = 'backward not autograd-safe (no update)'   # ... or backward raises (partial)
        with torch.no_grad():                          # validation loss (same loss form)
            k = ep % cv_input.shape[0]
            rkv = RobustKalmanOrig(sysmdl, cv_input[k], 1e-3, False, True,
                                   input_feat_mode=3, hidden_layers=TEACHER_HIDDEN, sl_model=0)
            rkv.nn = net; net.previous_output = torch.zeros(1, 1)
            outv = rkv.fnREKF(train=False)
            vloss = crit(outv[0][:, :-1], cv_target[k]).item()
        tr_hist.append(to_dB(loss.item())); val_hist.append(to_dB(vloss))
    dw = (torch.cat([p.detach().flatten() for p in net.parameters()]) - w0).norm().item()
    print(f"  teacher[{scen:7s}] weight change after {TEACHER['epochs']} epochs: dw = {dw:.2e}"
          f"   ({note} => the MLP net is NOT gradient-trainable)")
    history[('teacher RKNet', scen)] = {'train_dB': tr_hist, 'val_dB': val_hist}

    # evaluate the (unchanged) net on the shared test set at both inits, using rk.Xn like the
    # other methods; teacher [aligned] should land near 'existing Robust KNet' (both untrained MLP).
    for label, x0 in [('0init', torch.zeros(m, 1)), ('aligned', X0)]:
        sysmdl.m1x_0 = x0
        mses = []
        for k in range(test_input.shape[0]):
            rk = RobustKalmanOrig(sysmdl, test_input[k], 1e-3, False, True,
                                  input_feat_mode=3, hidden_layers=TEACHER_HIDDEN, sl_model=0)
            rk.nn = net; net.previous_output = torch.zeros(1, 1)
            with torch.no_grad(): rk.fnREKF(train=False)
            mses.append(crit(rk.Xn, test_target[k]).item())
        rec(f'teacher RKNet [{label}]', scen, to_dB(float(np.mean(mses))), f'(dw={dw:.1e})')
    sysmdl.m1x_0 = saved_init

if INCLUDE_TEACHER:
    for scen, sysmdl in [('full', sys_full), ('partial', sys_part)]:
        teacher_run(sysmdl, scen)
else:
    print('  teacher RKNet         [skipped]  (INCLUDE_TEACHER=False)')

NameError: name 'INCLUDE_TEACHER' is not defined

## Training history (train mode)

Per-epoch **training loss** (and **validation loss**) for each trainable model,
captured during the runs above and plotted below. Shown only in `train` mode — in
`eval` mode the models are loaded from checkpoints, so there is no training curve.

- **KalmanNet**: the curve is `NNTrain`'s returned per-epoch train/validation MSE
  (the optimizer/backward loop lives in `Pipelines/Pipeline_EKF.py`); it is also
  printed epoch-by-epoch above.
- **proposed Robust KalmanNet**: the curve is recorded inside its own optimization
  loop (in the *Metric & per-method helpers* cell) — training loss from the batch
  BPTT step, validation loss from a `no_grad` pass over the CV set each epoch.

In [ ]:
# Training history (train mode only). One panel per trained model x scenario.
if not history:
    print('No training history to show: RUN_MODE="eval" (models loaded from checkpoint), '
          'or no trainable model was enabled this run.')
else:
    mdl_names = [mm for mm in ['KalmanNet', 'proposed Robust KNet', 'teacher RKNet']
                 if any(k[0] == mm for k in history)]
    fig, axes = plt.subplots(len(mdl_names), 2, figsize=(11, 3.3 * len(mdl_names)), squeeze=False)
    for r, mdl in enumerate(mdl_names):
        for cix, scen in enumerate(['full', 'partial']):
            ax = axes[r][cix]; hh = history.get((mdl, scen))
            if hh is None:                      # scenario not trained this run
                ax.axis('off'); continue
            ep = range(1, len(hh['train_dB']) + 1)
            ax.plot(ep, hh['train_dB'], color='tab:blue', label='train')
            if hh.get('val_dB') is not None:
                ax.plot(ep, hh['val_dB'], color='tab:orange', ls='--', label='validation')
            ax.set_title(f'{mdl}  [{scen}]')
            ax.set_xlabel('epoch / step'); ax.set_ylabel('MSE [dB]')
            ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.suptitle(f'Training history  (RUN_MODE={RUN_MODE}, CONFIG={CONFIG})')
    fig.tight_layout(); plt.show()

## 5. Results table and the four qualitative orderings

In [ ]:
order = ['EKF', 'Robust EKF', 'KalmanNet', 'existing Robust KNet',
         'teacher RKNet [0init]', 'teacher RKNet [aligned]', 'proposed Robust KNet']
present = [mth for mth in order if mth in results]                       # skip disabled / absent methods
scens = ['full', 'partial']
df = pd.DataFrame({sc: {mth: results[mth].get(sc) for mth in present} for sc in scens})
print(df.to_string(float_format=lambda x: f'{x:+.3f}'))

def side(a, b, sc):
    if a not in results or b not in results or sc not in results[a] or sc not in results[b]:
        return f'{a} vs {b}: n/a (a model was skipped)'
    va, vb = results[a][sc], results[b][sc]
    return f'{a} ({va:+.2f}) {"<" if va < vb else ">"} {b} ({vb:+.2f})   [{"OK" if (va<vb) else "no"}]'
print('\n--- expected orderings (lower dB = better) ---')
print('FULL    EKF <= RobustEKF          :', side('EKF', 'Robust EKF', 'full'))
print('FULL    KalmanNet <= proposedRKNet:', side('KalmanNet', 'proposed Robust KNet', 'full'))
print('PARTIAL RobustEKF <  EKF          :', side('Robust EKF', 'EKF', 'partial'))
print('PARTIAL proposedRKNet <  KalmanNet:', side('proposed Robust KNet', 'KalmanNet', 'partial'))

print('\n--- my solution vs teacher (proposed vs teacher, aligned init) ---')
print('FULL    proposed <  teacher       :', side('proposed Robust KNet', 'teacher RKNet [aligned]', 'full'))
print('PARTIAL proposed <  teacher       :', side('proposed Robust KNet', 'teacher RKNet [aligned]', 'partial'))

# sanity: teacher [aligned] ~ existing Robust KNet (both are the untrained MLP -> training was a no-op)
for sc in scens:
    if 'teacher RKNet [aligned]' in results and 'existing Robust KNet' in results:
        ta, ex = results['teacher RKNet [aligned]'].get(sc), results['existing Robust KNet'].get(sc)
        if ta is not None and ex is not None:
            print(f'SANITY  teacher[aligned] ~ existing [{sc:7s}]: {ta:+.2f} vs {ex:+.2f} dB  (|Δ|={abs(ta-ex):.2f})')
df

## Reading the results

This notebook is the complete Task-4 pipeline: five estimators, two information
regimes, one shared test set, all trained/evaluated inside the notebook. The table
and the ordering checks above report exactly what the code produced - interpret and
decide next steps from these numbers.

Reference expectations from `docs/TASK4_DESIGN.md`: on matched (full) data robustness
is a mild penalty, so `EKF >= RobustEKF` and `KalmanNet >= RobustKalmanNet`; under the
`f`-mismatch, covariance robustness should help the EKF family (`RobustEKF > EKF`). The
proposed method's partial-info standing versus KalmanNet, and its learned `c`, are read
directly from the table above. To scale precision: raise `args.n_steps` (KalmanNet) and
`PROP['epochs']` (proposed); to explore the noise axis, sweep `r2` as the paper does.